In [19]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os

import sys
from PyQt5 import QtWidgets


os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())
from config import DATA_DIR,INPUT_DIR
from typing import Dict, Any


# def slice_group_data(raw_group_data, start, end):
#     """
#     从 raw_group_data 中裁剪时间区间 [basicSa, end)
#     """
#     return {
#         step: raw_group_data[step]
#         for step in range(start, end)
#         if step in raw_group_data
#     }

from src.io.operate_group_data import slice_group_data

import src.io.operate_group_data as operate_group_data
import src.io.read_snap_xml  as read_snap_xml
from src.config.viewer_config import G60_CONFIG



backend (before pyplot): QtAgg
backend (after pyplot): QtAgg


In [2]:
import importlib
import src.io.read_snap_xml  as read_snap_xml
importlib.reload(read_snap_xml)


<module 'src.io.read_snap_xml' from 'E:\\paper11\\generic\\src\\io\\read_snap_xml.py'>

In [3]:
from src.viz.pyqt_main2 import SatelliteViewer


from PyQt5 import QtWidgets

In [4]:
import importlib
import src.viz.pyqt_main2 as pyqt_main2

importlib.reload(pyqt_main2)

SatelliteViewer = pyqt_main2.SatelliteViewer

In [5]:
import importlib
import src.io.operate_group_data as operate_group_data

importlib.reload(operate_group_data)

slice_group_data = operate_group_data.slice_group_data


In [6]:
import importlib
import src.viz.group_data_transform as group_data_transform

importlib.reload(group_data_transform)


<module 'src.viz.group_data_transform' from 'E:\\paper11\\generic\\src\\viz\\group_data_transform.py'>

In [7]:
import importlib
import src.config.constellation_config as constellation_config

importlib.reload(constellation_config)


<module 'src.config.constellation_config' from 'E:\\paper11\\generic\\src\\config\\constellation_config.py'>

In [11]:
from pathlib import Path

DATA_DIR = Path(r"E:\paper11")
BASEDIR =  DATA_DIR / "data"




xml_file = BASEDIR /'basic_file'/ 'satellitesposition' / "station_visible_satellites_20250106.xml"
# xml_file = DATA_DIR / "visibile_data" / "G60" / "g60.xml"

In [9]:
Topology_DIR = 'topology_design'

In [ ]:
# from pathlib import Path
#
# DATA_DIR = Path(r"C:\usrspace\mywork\data_paper2")
# BASEDIR =  DATA_DIR / "visibile_data"
#
# VERSION1="test"
#
# xml_file = BASEDIR / VERSION1 / "visibility_snapshots.xml"


In [ ]:
# from pathlib import Path
#
# DATA_DIR = Path(r"C:\usrspace\mywork\data_paper2")
# BASEDIR =  DATA_DIR / "visibile_data"
#
# VERSION1="DATA"
#
# xml_file = BASEDIR / VERSION1 / "GW.xml"


In [ ]:
%%sql


In [12]:
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

# ====================== 读取数据 ======================

# start_ts = 10717
# # end_ts   = 86399
# end_ts   = 11640
# # 解析 XML 得到 group_data，结构：{time_step: {'groups': {...}}}
# group_data = read_snap_xml.parse_xml_group_data(xml_file, start_ts, end_ts)
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 86164
# 注意，这里是一个恒星日
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, G60_CONFIG,RAW_START, RAW_END)

#下面是图变换的。
#

In [22]:
# 这里再进行小区间分开，实际上也是进行快速迭代
# 用法（左闭右开
start_ts =0

end_ts =86164

group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [15]:
base_groupid_now = 0

In [23]:
from src.config.constellation_config import ConstellationConfig
constellation_config = ConstellationConfig(
    name="G60",
    N=36,
    P=18,

)
rev_group_data,offset = group_data_transform.modify_group_data(group_data,constellation_config, base_groupid=base_groupid_now)

In [17]:
import importlib
import src.config.viewer_config as vc
importlib.reload(vc)
from src.config.viewer_config import G60_CONFIG

下面主要是为了测试检测我们的图

In [20]:
from src.config.viewer_config import G60_CONFIG
# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []

In [24]:


# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data, G60_CONFIG)
viewer.setWindowTitle("group_data")
viewer.resize(1200, 700)
# viewer.edges_by_step =
# viewer.pending_links_by_step =
viewer.show()



In [26]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(rev_group_data,config=G60_CONFIG)
viewer.setWindowTitle("rev_group_data")
viewer.resize(1200, 700)
# viewer.edges_by_step =
# viewer.pending_links_by_step =
viewer.show()


In [ ]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

## 绘制motif
我们的motif代码，实际上除了同构图绘制外，在原图上也可以设计的
## motif 配置
这一章节，我们就要用多种拓扑motif了，注意了，motif的配置，无需在变换的拓扑上操作，直接在原来的图上操作即可，

In [25]:
Topology_Version = 'grid_full'

In [26]:
# from draw.basic_functio.topology_config import TopologyRecorder
from src.model.topology_config import  TopologyRecorder
P, N = 18, 36
rec = TopologyRecorder(P, N)

# 这里的 nodes 只是为了兼容你现有 API，真正生成边时会用 rec 里录下来的 motif
nodes = {}

## 下面是各种motif方案

In [30]:
#grid +
rec.write_distinct_motif(
            p_start=0, p_end=17,
            y_start=0, y_end=35,
           nodes=nodes,
                         option=0,
        )
rec.write_distinct_motif(
            p_start=0, p_end=17,
            y_start=0, y_end=35,
           nodes=nodes,
                         option=1,
        )
rec.write_distinct_motif(
            p_start=0, p_end=17,
            y_start=0, y_end=35,
           nodes=nodes,
                         option=2,
        )


In [31]:

# # 4) 如果你想看一段时间内每个 step 的 motif 边
all_adj = rec.render_adj_range(
    t_start_incl=0,
    t_end_excl=10,
    eval_env={"start_ts": 0, "end_ts": 10},
)

# 写入motif
rec.base_groupid = 0    # 静态拓扑不涉及同构变换，给默认值跳过检查
rec.save(BASEDIR / Topology_DIR / Topology_Version/"config" / "motif.json")



In [32]:
## 读取motif
## 这一段仅仅是为了演示读取config文件而已

from draw.basic_functio.topology_config import load_config

cfg = load_config(BASEDIR / Topology_DIR / Topology_Version/"config" / "motif.json")
# 重建 recorder，把 motif 列表灌进去
rec = TopologyRecorder(cfg.P, cfg.N)
rec._motifs = cfg.motifs
inter_adj = rec.render_adj_at(t=0, eval_env={"start_ts": 0, "end_ts": 1})


viewer = SatelliteViewer( group_data,G60_CONFIG)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = {step: inter_adj for step in range(start_ts, end_ts)}


viewer.show()
_viewer_list.append(viewer)


In [33]:
def build_full_option_adj(P, N, options=(0, 1, 2, 4), *, skip_first_last_source=False):
    """
    生成全候选 inter-plane 链接邻接表，格式适配 SatelliteViewer.edges_by_step。

    option 含义：
      0: (p, y) -> (p+1, y)
      1: (p, y) -> (p+1, y-1)
      2: (p, y) -> (p+2, y)
      4: (p, y) -> (p+1, y+1)

    返回:
      adj[src_id] = set(dst_id)
    """
    option_deltas = {
        0: (1, 0),
        1: (1, -1),
        2: (2, 0),
        4: (1, 1),
    }

    adj = {}

    for p in range(P):
        if skip_first_last_source and p in (0, P - 1):
            continue

        for y in range(N):
            src = p * N + y

            for option in options:
                dp, dy = option_deltas[option]
                q = p + dp

                if not (0 <= q < P):
                    continue

                yy = (y + dy) % N
                dst = q * N + yy

                adj.setdefault(src, set()).add(dst)

    return adj


def repeat_static_adj_for_viewer(static_adj, group_data):
    """
    把静态邻接表复制成 viewer.edges_by_step 需要的 all_adj。
    这里每个 step 引用同一个 static_adj，不会大量复制内存。
    """
    return {int(step): static_adj for step in group_data.keys()}

In [34]:
P = G60_CONFIG.P
N = G60_CONFIG.N

full_adj = build_full_option_adj(
    P,
    N,
    options=(0, 1, 2, 4),
    skip_first_last_source=False,  # 如果首轨/末轨不作为源轨，改 True
)

all_adj = repeat_static_adj_for_viewer(full_adj, group_data)

viewer = SatelliteViewer(group_data, G60_CONFIG)
viewer.setWindowTitle("full option topology")
viewer.resize(1200, 700)
viewer.edges_by_step = all_adj

viewer.show()
_viewer_list.append(viewer)

In [35]:
import sys
from pathlib import Path
from PyQt5 import QtWidgets

sys.path.insert(0, str(Path(r"E:\paper11\generic")))

from paper1notebook.betweenness_time_viewer import (
    EdgeBetweennessTimeViewer as SatelliteViewer,
    build_betweenness_edges_by_step,
)

group_data_100 = {
    step: group_data[step]
    for step in range(0, 101)
    if step in group_data
}

all_adj, summaries = build_betweenness_edges_by_step(
    group_data_100,
    G60_CONFIG,
    group_a=2,
    group_b=3,
    steps=range(0, 101),
    max_draw_edges=500,
    min_count=1,
)

app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

viewer = SatelliteViewer(group_data_100, G60_CONFIG)
viewer.setWindowTitle("group2-group3 edge betweenness 0-100s")
viewer.resize(1200, 700)
viewer.edges_by_step = all_adj
viewer.betweenness_summaries = summaries

viewer.show()
viewer.plot_satellites(0)

_viewer_list.append(viewer)

In [41]:
viewer = SatelliteViewer( group_data,G60_CONFIG)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = all_adj

viewer.show()
_viewer_list.append(viewer)
# viewer.show_envelopes_static(
#     rects_by_group=rects,
#     expand=0.35,
#     colors=colors,
#     persist=True
# )

In [ ]:

# 注意上述我们是在同构拓扑序列上进行的，因此，我们还要将同构拓扑序列进行还原，同时，我们还要考虑到建链时间约束
# import draw.basic_functio.revdata2rawdata as revdata2rawdata
# # attention ,here  it just composed of the inter-link, intra_link hasn't benn conclued
# raw_inter_edge = revdata2rawdata.revedge2rawedge(all_rev_inter_edge,offset)

In [ ]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

我们要处理好建链时间冲突，因此，下面就是处理冲突的代码

注意，这里其实就是IG 内部要处理的建联图

In [ ]:

# 下面是把边转为node存储，因为这种方式存储会比较方便

##
# import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
# all_nodes = inter_edge2nodes.trans_edge2node(raw_inter_edge,P,N)

In [ ]:
## motif设计
# motif设计的原则是,

In [20]:
import src.model.basiclink as basiclink

In [21]:
# 1) 静态 inter 拓扑：只构一次（不要 render_adj_range）
inter_once = rec.render_adj_at(
    t=start_ts,
    eval_env={"start_ts": start_ts, "end_ts": end_ts}
)

# 2) 如果你当前走的是“rev -> raw”流程，用你现有函数转一次
# raw_inter_once = revdata2rawdata.revedge2rawedge({start_ts: inter_once}, offset, N)[start_ts]

# 如果你已经是 raw 坐标设计，直接用 inter_once
raw_inter_once = inter_once

# 3) 你现有的双向化函数，复用
raw_inter_once = basiclink.make_edges_bidirectional(raw_inter_once)

# 4) 静态总图 = intra ring + static inter
base_neighbors = {
    i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
    for i in range(P) for j in range(N)
}
static_edges = {node: {r, l} for node, (r, l) in base_neighbors.items()}
for src, dsts in raw_inter_once.items():
    static_edges.setdefault(src, set()).update(dsts)

# 5) 按现有 API 组装 step->adj（每个 step 共用同一张图）
all_edges = {step: static_edges for step in range(start_ts, end_ts)}

In [22]:

# 4) 画图验证
viewer = SatelliteViewer(group_data, G60_CONFIG)
viewer.setWindowTitle("verify all_edges_view")
viewer.resize(1400, 800)
viewer.edges_by_step = all_edges
viewer.show()
_viewer_list.append(viewer)



In [23]:
# 获取station pairs，
# 这个一般只要运行一次就可以了
WIN_START = RAW_START
WIN_END = RAW_START + 99   # 100秒窗口（包含 WIN_START 和 WIN_END）


# series = read_snap_xml.parse_station_timeseries(
#     xml_file, [5, 7, 9, 18, 15, 19], WIN_START, WIN_END
# )


In [27]:
# S6 = series[1]
# S8 = series[5]
import  importlib
import  src.model.plot_2city_shortest_path as plot_2city_shortest_path
importlib.reload(plot_2city_shortest_path)

<module 'src.model.plot_2city_shortest_path' from 'D:\\paper3\\generic\\src\\model\\plot_2city_shortest_path.py'>

In [26]:
# 区域定义（从 G60_CONFIG.station_groups 读取）
all_regions = {}
for gid, info in G60_CONFIG.station_groups.items():
    all_regions[gid] = info["stations"]


In [34]:
# =========================
# A) 循环外：准备 region station 列表 + 一次性解析 series
# =========================
import src.model.plot_2city_shortest_path as plot_2city_shortest_path
from src.config.viewer_config import G60_CONFIG
from src.io import read_snap_xml

# 收集所有 station id
all_station_ids = sorted(set(
    sid for stations in all_regions.values() for sid in stations
))

# 如果你要严格用 1..11 和 12..21，请改成：
# region_a_stations = list(range(1, 12))
# region_b_stations = list(range(12, 22))

# 你的 TOTAL_END 在外层循环是“右开”，所以这里用 TOTAL_END-1 做闭区间
series_list = read_snap_xml.parse_station_timeseries(
    xml_file, all_station_ids, RAW_START, RAW_END - 1
)
series_by_station = {sid: ts for sid, ts in zip(all_station_ids, series_list)}



In [35]:
len(series_by_station)

31

In [36]:
## 下列都是测试
# 包括 计算最短路径，可视化
from src.io import read_snap_xml
# from src.model.plot_2city_shortest_path import compute_stationpair_min_hops_over_time
from src.viz.pyqt_main2 import SatelliteViewer
from src.config.viewer_config import ViewerConfig, G60_CONFIG

# 你已有变量：xml_file, RAW_START, all_edges, _viewer_list

# =========================
# 1) 只取 100 秒窗口
# =========================
WIN_START = RAW_START
WIN_END = RAW_START + 99   # 含端点，共100个step（若1step=1s）

# 只截取窗口内拓扑，避免 viewer 和计算时间轴不一致
sub_edges = {t: all_edges.get(t, {}) for t in range(WIN_START, WIN_END + 1)}
S6=series_by_station[2]
S8=series_by_station[20]
# =========================
# 2) 计算 S6-S8 最短跳数（每步取接入对中的最小）
# =========================
df_s6_s8 = plot_2city_shortest_path.compute_stationpair_min_hops_over_time(
    all_edges=sub_edges,
    paris=S6,
    chongqin=S8,
    steps=(WIN_START, WIN_END),
    undirected=True,
    return_pair=True,
    return_path=True,
    static_topology=True,
    precompute_all_pairs=True,
)

print(df_s6_s8[["time", "min_shortest_path", "best_s", "best_d", "path"]].head(20))



# 1) 导出原始CSV（含最短路对和路径）
# csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
#     all_edges=sub_edges,
#     paris=S6,
#     chongqin=S8,
#     out_dir=FIGURE_DIR,
#     basename="s6_s8_100s_raw",
#     steps=(WIN_START, WIN_END),
#     undirected=True,
#     static_topology=True,
#     precompute_all_pairs=True,
#     with_pair=True,
#     with_path=True,
# )
# print("CSV:", csv_path)

# 2) 导出一张图并显示（初步核对）
# fig, ax, df_plot = plot_2city_shortest_path.plot_stationpair_min_hops_over_time(
#     all_edges=sub_edges,
#     paris=S6,
#     chongqin=S8,
#     steps=(WIN_START, WIN_END),
#     undirected=True,
#     static_topology=True,
#     precompute_all_pairs=True,
#     title="S6-S8 minimal shortest hops (100s)",
#     show=True,
#     save=True,
#     save_dir=FIGURE_DIR,
#     basename="s6_s8_100s_plot",
#     formats=("png",),
#     return_handles=True,
# )


# =========================
# 3) 构造仅用于可视化的 group_data（只显示 S6/S8 两组）
# =========================
# group_data_s6_s8 = {}
# for t in range(WIN_START, WIN_END + 1):
#     s6_set = set(S6.get(t, set()))
#     s8_set = set(S8.get(t, set()))
#     group_data_s6_s8[t] = {
#         "groups": {
#             0: s6_set,   # 组0 -> S6
#             1: s8_set,   # 组1 -> S8
#         },
#         "all_mentioned": s6_set | s8_set
#     }
group_data_s6_s8 = operate_group_data.build_stationpair_group_data(S6, S8, WIN_START, WIN_END)

# 可选：做一个仅S6/S8的viewer配置，让图例更清楚
PAIR_CFG = ViewerConfig(
    name="S6_S8_VERIFY",
    N=G60_CONFIG.N,
    P=G60_CONFIG.P,
    station_groups={
        0: {"name": "S6", "stations": [5]},
        1: {"name": "S8", "stations": [7]},
    },
    group_colors=["#ff4d4f", "#2f54eb"],
)

# =========================
# 4) 把最短路叠加到 viewer（蓝色路径）
# =========================
path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]

viewer = SatelliteViewer(group_data_s6_s8, PAIR_CFG)
viewer.setWindowTitle("verify S6-S8 shortest path (100s)")
viewer.resize(1400, 800)
viewer.edges_by_step = sub_edges
viewer.set_paths(path_by_step)   # 叠加每步最短路（蓝线）

viewer.show()
_viewer_list.append(viewer)


    time  min_shortest_path best_s best_d  \
0      0               11.0    144    227   
1      1               11.0    144    227   
2      2               11.0    144    227   
3      3               11.0    144    227   
4      4               11.0    144    227   
5      5               11.0    144    227   
6      6               11.0    144    227   
7      7               11.0    144    227   
8      8               11.0    144    227   
9      9               11.0    144    227   
10    10               11.0    144    227   
11    11               11.0    144    227   
12    12               11.0    144    227   
13    13               11.0    144    227   
14    14               11.0    144    227   
15    15               11.0    144    227   
16    16               11.0    144    227   
17    17               11.0    144    227   
18    18               11.0    144    227   
19    19               11.0    144    227   

                                                 path 

In [37]:
path_by_step[36]

[108, 73, 74, 75, 76, 113, 114, 151, 152, 189, 190, 227]

## 正式批量导出
下列是在jupeter里尝试先批量导出,后续，juper只作为拓扑设计以及小时间段内的数据验证，批量导出交由py程序实现

In [28]:
# =========================
# A) 循环外：准备 region station 列表 + 一次性解析 series
# =========================
import src.model.plot_2city_shortest_path as plot_2city_shortest_path
from src.config.viewer_config import G60_CONFIG
from src.io import read_snap_xml

region_a_gid = 0   # region1
region_b_gid = 1   # region2

region_a_stations = list(G60_CONFIG.station_groups[region_a_gid]["stations"])
region_b_stations = list(G60_CONFIG.station_groups[region_b_gid]["stations"])

# 如果你要严格用 1..11 和 12..21，请改成：
# region_a_stations = list(range(1, 12))
# region_b_stations = list(range(12, 22))

all_station_ids = sorted(set(region_a_stations) | set(region_b_stations))

# 你的 TOTAL_END 在外层循环是“右开”，所以这里用 TOTAL_END-1 做闭区间
series_list = read_snap_xml.parse_station_timeseries(
    xml_file, all_station_ids, RAW_START, RAW_END - 1
)
series_by_station = {sid: ts for sid, ts in zip(all_station_ids, series_list)}

print("regionA stations:", region_a_stations)
print("regionB stations:", region_b_stations)
print("total station pairs:", len(region_a_stations) * len(region_b_stations))


regionA stations: [0, 1, 2, 3, 4]
regionB stations: [5, 6, 7, 8, 9, 10]
total station pairs: 30


In [ ]:

# 3) 批量导出 CSV：region1 全部 x region2 全部
OUT_DIR = Path(FIGURE_DIR) / f"region1_to_region2_{WIN_START}_{WIN_END}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for s1 in region_a_stations:
    for s2 in region_b_stations:
        # 你要的命名格式（建议文件名用 --，兼容性更好）
        # 例：region1--station1-region2--station7.csv
        basename = f"region1--station{s1}-region2--station{s2}"

        csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
            all_edges=sub_edges,
            paris=series_by_station[s1],
            chongqin=series_by_station[s2],
            out_dir=OUT_DIR,
            basename=basename,
            steps=(WIN_START, WIN_END),
            undirected=True,
            static_topology=True,
            precompute_all_pairs=True,
            with_pair=True,
            with_path=False,
        )
        print("[OK]", csv_path)

# 4) 初步查看：抽一对画图（不影响批量导出）
# probe_s1 = region1_stations[0]
# probe_s2 = region2_stations[0]
# plot_stationpair_min_hops_over_time(
#     all_edges=sub_edges,
#     paris=series_by_station[probe_s1],
#     chongqin=series_by_station[probe_s2],
#     steps=(WIN_START, WIN_END),
#     undirected=True,
#     static_topology=True,
#     precompute_all_pairs=True,
#     title=f"region1-station{probe_s1} to region2-station{probe_s2}",
#     show=True,
#     save=True,
#     save_dir=OUT_DIR,
#     basename=f"preview_region1--station{probe_s1}-region2--station{probe_s2}",
#     formats=("png",),
# )


[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station12.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station13.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station14.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station15.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station16.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station17.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station18.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station19.csv
[OK] D:\paper3\data\satellitesposition\path\region1_to_region2_0_99\region1--station0-region2--station20.csv
[OK] D:\paper3\data

In [31]:
import  src.model.static_hop_table  as static_hop_table
importlib.reload(static_hop_table)

<module 'src.model.static_hop_table' from 'D:\\paper3\\generic\\src\\model\\static_hop_table.py'>

In [32]:
# from src.model.static_hop_table import (
#     build_static_graph, precompute_hop_matrix,
#     compute_region_pair_timeseries, export_pair_csvs
# )
from src.config.viewer_config import G60_CONFIG
from src.io import read_snap_xml
from pathlib import Path
import time
from datetime import datetime

# =========================
# 日志工具
# =========================
T0 = time.perf_counter()
def log(msg: str):
    now = datetime.now().strftime("%H:%M:%S")
    dt = time.perf_counter() - T0
    print(f"[{now} | +{dt:8.2f}s] {msg}", flush=True)

# =========================
# 参数
# =========================
WIN_START = 0
WIN_END = 100   # parse_station_timeseries 是“含 end_step”，这里是 101 个 step（0~100）
DIST_METHOD = "floyd"  # "floyd" 或 "all_pairs_bfs"
sub_edges = {t: all_edges.get(t, {}) for t in range(WIN_START, WIN_END + 1)}

log("Step 1/7: 读取 region 配置")
region1 = list(G60_CONFIG.station_groups[0]["stations"])
region2 = list(G60_CONFIG.station_groups[1]["stations"])
log(f"region1 stations={region1}")
log(f"region2 stations={region2}")

all_station_ids = sorted(set(region1) | set(region2))
log(f"合并后 station 总数={len(all_station_ids)}")

# 可选：检查 sub_edges 覆盖情况
expect_steps = set(range(WIN_START, WIN_END + 1))
have_steps = set(sub_edges.keys())
missing = sorted(expect_steps - have_steps)
if missing:
    log(f"警告：sub_edges 缺少 {len(missing)} 个 step，示例: {missing[:10]}")
else:
    log("sub_edges 时间窗覆盖完整")

# =========================
# 读取站点时序
# =========================
log("Step 2/7: 开始解析 station timeseries")
t = time.perf_counter()
series_list = read_snap_xml.parse_station_timeseries(
    xml_file, all_station_ids, WIN_START, WIN_END
)
series_by_station = {sid: ts for sid, ts in zip(all_station_ids, series_list)}
log(f"station timeseries 解析完成，耗时 {time.perf_counter() - t:.2f}s")

# =========================
# 构建静态图
# =========================
log("Step 3/7: 构建静态拓扑图")
t = time.perf_counter()
G = static_hop_table.build_static_graph(
    sub_edges, total_sats=G60_CONFIG.total_sats, undirected=True
)
log(f"图构建完成: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, 耗时 {time.perf_counter() - t:.2f}s")

# =========================
# 预计算最短跳数表
# =========================
log(f"Step 4/7: 开始预计算全点对最短跳数（method={DIST_METHOD}）")
t = time.perf_counter()


dist, next_hop = static_hop_table.precompute_hop_and_next_hop(G, G60_CONFIG.total_sats)
log(f"{DIST_METHOD} 执行完毕，dist shape={dist.shape}，耗时 {time.perf_counter() - t:.2f}s")

# =========================
# 批量 station-pair 查表
# =========================
pairs = [(a, b) for a in region1 for b in region2]
log(f"Step 5/7: 开始读取 station pair 状态并查表，pair 数={len(pairs)}")

t = time.perf_counter()


df_all = static_hop_table.compute_region_pair_timeseries(
    dist=dist,
    next_hop=next_hop,
    series_by_station=series_by_station,
    station_pairs=pairs,
    steps=range(WIN_START, WIN_END + 1),
    left="region1",
    right="region2",
)



log(f"station pair 查表完成，结果行数={len(df_all)}，耗时 {time.perf_counter() - t:.2f}s")

# =========================
# 导出
# =========================
out_dir = Path(FIGURE_DIR) / f"region1_to_region2_{WIN_START}_{WIN_END}"
log(f"Step 6/7: 开始导出 CSV 到 {out_dir}")

t = time.perf_counter()
csv_paths = static_hop_table.export_pair_csvs(df_all, out_dir, left="region1", right="region2")


log(f"导出完成，共 {len(csv_paths)} 个文件，耗时 {time.perf_counter() - t:.2f}s")

# =========================
# 汇总
# =========================
log("Step 7/7: 任务结束，展示前几行")
print(df_all.head())
print("exported:", len(csv_paths))
print("first 5 files:", [str(p) for p in csv_paths[:5]])


[16:41:52 | +    0.00s] Step 1/7: 读取 region 配置
[16:41:52 | +    0.00s] region1 stations=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[16:41:52 | +    0.01s] region2 stations=[12, 13, 14, 15, 16, 17, 18, 19, 20]
[16:41:52 | +    0.01s] 合并后 station 总数=21
[16:41:52 | +    0.01s] sub_edges 时间窗覆盖完整
[16:41:52 | +    0.01s] Step 2/7: 开始解析 station timeseries
[16:42:29 | +   36.68s] station timeseries 解析完成，耗时 36.67s
[16:42:29 | +   36.69s] Step 3/7: 构建静态拓扑图
[16:42:29 | +   36.69s] 图构建完成: nodes=648, edges=1260, 耗时 0.00s
[16:42:29 | +   36.70s] Step 4/7: 开始预计算全点对最短跳数（method=floyd）
[16:42:30 | +   37.52s] floyd 执行完毕，dist shape=(648, 648)，耗时 0.82s
[16:42:30 | +   37.52s] Step 5/7: 开始读取 station pair 状态并查表，pair 数=108
[16:42:30 | +   38.02s] station pair 查表完成，结果行数=10908，耗时 0.50s
[16:42:30 | +   38.02s] Step 6/7: 开始导出 CSV 到 D:\paper3\data\satellitesposition\static_path\region1_to_region2_0_100
[16:42:30 | +   38.21s] 导出完成，共 108 个文件，耗时 0.19s
[16:42:30 | +   38.21s] Step 7/7: 任务结束，展示前几行
   time  station_a  sta

In [ ]:


#xiamianshi meiyouyiyi de
viewer = SatelliteViewer(new_group_data)
viewer.setWindowTitle("groupdata with rawedge")

viewer.resize(1200, 700)
viewer.edges_by_step =all_edges

viewer.show()


In [ ]:
# 某些轨道之间的
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path






basename = f"avgspath_g0_4_baseline_{start_ts}_to_{end_ts}"


csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
    all_edges=all_edges,            # 传入切片后的边
    group_data=new_group_data,      # 传入切片后的组数据
    out_dir=FIGURE_DIR,
    basename=basename,
    group_a=0,
    group_b=1,
    steps=(start_ts, end_ts), # 明确指定当前处理的范围
    undirected=True
)


In [ ]:
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path

for i in range(3):
    for j in range(3):
        # 为每个 station 指定选择第几个轨道（0-based index）
        ORBIT_SELECTION = {
            0: i,  # Station 0 选第 1 个轨道（0-based，即第二个）
            1: j,  # Station 1 选第 1 个轨道
        }

        new_group_data = filter_group_data_by_orbit(group_data, ORBIT_SELECTION, N,debug=True)

        basename = f"avgspath_g0_4_baseline_{i}_to_{j}"


        csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
            all_edges=all_edges,                # 传入切片后的边
            group_data=new_group_data,      # 传入切片后的组数据
            out_dir=FIGURE_DIR,
            basename=basename,
            group_a=0,
            group_b=1,
            steps=(start_ts, end_ts), # 明确指定当前处理的范围
            undirected=True
        )


In [ ]:
# 某些轨道之间的
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path






basename = f"avgspath_g0_4_baseline_{start_ts}_to_{end_ts}"


csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
    all_edges=all_edges,            # 传入切片后的边
    group_data=group_data,      # 传入切片后的组数据
    out_dir=FIGURE_DIR,
    basename=basename,
    group_a=0,
    group_b=1,
    steps=(start_ts, end_ts), # 明确指定当前处理的范围
    undirected=True
)


In [ ]:
group_data

In [ ]:
import time
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
# ================= 1. 设置测试参数 =================
TOTAL_START = start_ts
TOTAL_END = end_ts     # 测试总长度
CHUNK_SIZE = 10000    # 切片大小
# 你的输出路径


# 确保文件夹存在
if not FIGURE_DIR.exists():
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== 开始顺序测试: 范围 {TOTAL_START}-{TOTAL_END}, 分片大小 {CHUNK_SIZE} ===")

# ================= 2. 顺序循环执行 =================
for batch_start in range(TOTAL_START, TOTAL_END, CHUNK_SIZE):

    # 计算当前结束点
    batch_end = min(batch_start + CHUNK_SIZE, TOTAL_END)

    print(f"\n>> 正在处理分片: {batch_start} 到 {batch_end} ...")

    # --- A. 切片 all_edges (提取当前 100s 的拓扑) ---
    target_steps = range(batch_start, batch_end)
    sub_edges = {
        step: all_edges[step]
        for step in target_steps
        if step in all_edges
    }

    # 检查数据是否为空
    if not sub_edges:
        print(f"   [警告] 时间段 {batch_start}-{batch_end} 没有拓扑数据，跳过。")
        continue
    else:
        print(f"   [数据] sub_edges 包含 {len(sub_edges)} 个时刻。")

    # --- B. 切片 group_data (提取当前 100s 的组信息) ---
    # 假设 slice_group_data 是你之前定义好的函数
    sub_group_data = slice_group_data(raw_group_data, batch_start, batch_end)

    # --- C. 执行计算与导出 ---
    basename = f"avgspath_g0_4_baseline_{batch_start}_to_{batch_end}"

    try:
        csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
            all_edges=sub_edges,            # 传入切片后的边
            group_data=sub_group_data,      # 传入切片后的组数据
            out_dir=FIGURE_DIR,
            basename=basename,
            group_a=0,
            group_b=1,
            steps=(batch_start, batch_end), # 明确指定当前处理的范围
            undirected=True
        )
        print(f"   [成功] 文件已生成: {csv_path}")

    except Exception as e:
        print(f"   [错误] 计算时发生异常: {e}")
        # 如果出错，打印出堆栈以便调试
        import traceback
        traceback.print_exc()

print("\n=== 测试运行结束 ===")

In [ ]:
import time
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_2city_shortest_path as plot_2city_shortest_path

# ================= 1. 设置测试参数 =================
TOTAL_START = start_ts
TOTAL_END = end_ts     # 测试总长度
CHUNK_SIZE = 10000    # 切片大小
# 你的输出路径


# 确保文件夹存在
if not FIGURE_DIR.exists():
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== 开始顺序测试: 范围 {TOTAL_START}-{TOTAL_END}, 分片大小 {CHUNK_SIZE} ===")

# ================= 2. 顺序循环执行 =================
for batch_start in range(TOTAL_START, TOTAL_END, CHUNK_SIZE):

    # 计算当前结束点
    batch_end = min(batch_start + CHUNK_SIZE, TOTAL_END)

    print(f"\n>> 正在处理分片: {batch_start} 到 {batch_end} ...")

    # --- A. 切片 all_edges (提取当前 100s 的拓扑) ---
    target_steps = range(batch_start, batch_end)
    sub_edges = {
        step: all_edges[step]
        for step in target_steps
        if step in all_edges
    }

    # 检查数据是否为空
    if not sub_edges:
        print(f"   [警告] 时间段 {batch_start}-{batch_end} 没有拓扑数据，跳过。")
        continue
    else:
        print(f"   [数据] sub_edges 包含 {len(sub_edges)} 个时刻。")

    # --- B. 切片 group_data (提取当前 100s 的组信息) ---
    # 假设 slice_group_data 是你之前定义好的函数
    sub_group_data = slice_group_data(raw_group_data, batch_start, batch_end)

    # --- C. 执行计算与导出 ---
    # basename = f"avgspath_g0_4_baseline_{batch_start}_to_{batch_end}"

    try:
        # csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
        #     all_edges=sub_edges,            # 传入切片后的边
        #     group_data=sub_group_data,      # 传入切片后的组数据
        #     out_dir=FIGURE_DIR,
        #     basename=basename,
        #     group_a=0,
        #     group_b=1,
        #     steps=(batch_start, batch_end), # 明确指定当前处理的范围
        #     undirected=True
        # )

        citys1name = f"city22_{batch_start}_to_{batch_end}"
        csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
            sub_edges, S8, S16,
            out_dir=FIGURE_DIR,
            basename=citys1name,
            steps=(batch_start, batch_end),
            undirected=True,
            with_pair=True,
            with_path=False
        )

        # citys2name = f"city2_{batch_start}_to_{batch_end}"
        # csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
        #     sub_edges, S8, S16,
        #     out_dir=FIGURE_DIR,
        #     basename=citys2name,
        #     steps=(batch_start, batch_end),
        #     undirected=True,
        #     with_pair=True,
        #     with_path=False
        # )
        # citys3name = f"city3_{batch_start}_to_{batch_end}"
        # csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
        #     sub_edges, S10, S20,
        #     out_dir=FIGURE_DIR,
        #     basename=citys3name,
        #     steps=(batch_start, batch_end),
        #     undirected=True,
        #     with_pair=True,
        #     with_path=False
        # )




        print(f"   [成功] 文件已生成: {csv_path}")

    except Exception as e:
        print(f"   [错误] 计算时发生异常: {e}")
        # 如果出错，打印出堆栈以便调试
        import traceback
        traceback.print_exc()

print("\n=== 测试运行结束 ===")